# Free GPU talking-head worker
1. **Runtime → Change runtime type → T4 GPU**.
2. Run the cells in order (about 5-10 min the first time).
3. Copy the printed `TALKING_HEAD_URL` and `TALKING_HEAD_TOKEN` into your Render service's Environment tab.

The session ends after a few hours idle; re-run to get a new URL. Use only a face you own or an AI-generated fictional face, never a real person's likeness without consent.

In [ ]:
!git clone https://github.com/OpenTalker/SadTalker /content/SadTalker
%cd /content/SadTalker
!pip install -q fastapi uvicorn python-multipart
# SadTalker pins old libs; if the next line errors on newer Colab Python, see the SadTalker README "Colab" notes.
!pip install -q -r requirements.txt
!bash scripts/download_models.sh

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
!wget -q -O /content/head_worker.py https://raw.githubusercontent.com/<YOUR_GITHUB_USER>/<YOUR_REPO>/main/colab/head_worker.py
# ...or upload colab/head_worker.py to /content/ using the Files panel instead of the wget above.

In [ ]:
import os, secrets, subprocess, time, re
token = secrets.token_urlsafe(16)
os.environ["WORKER_TOKEN"] = token
os.environ["ENGINE_DIR"] = "/content/SadTalker"
subprocess.Popen("cd /content && python -m uvicorn head_worker:app --port 8001", shell=True)
time.sleep(5)
tun = subprocess.Popen(["/content/cloudflared","tunnel","--url","http://localhost:8001"], stderr=subprocess.PIPE, text=True)
url = None
while not url:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", tun.stderr.readline())
    url = m.group(0) if m else None
print("TALKING_HEAD_URL  =", url)
print("TALKING_HEAD_TOKEN =", token)